# Day 1: Neuron, Activations and the Forward Pass (NumPy)

Aaj ka goal: ek neuron, activation functions, aur 2-layer network ka forward pass NumPy me likhna.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

np.random.seed(42)

## 1. Activation functions

- **Sigmoid**: output 0 se 1. Binary classification ke output layer me use hota hai.
- **Tanh**: output -1 se 1, zero-centered. Purane hidden layers me use hota tha.
- **ReLU**: `max(0, z)`. Hidden layers ka default, fast aur vanishing gradient kam.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def tanh(z):
    return np.tanh(z)

z = np.linspace(-6, 6, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, (name, f) in zip(axes, [("Sigmoid", sigmoid), ("Tanh", tanh), ("ReLU", relu)]):
    ax.plot(z, f(z))
    ax.set_title(name)
    ax.axhline(0, color="gray", lw=0.5)
    ax.axvline(0, color="gray", lw=0.5)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Ek single neuron

`z = w . x + b`, phir `a = activation(z)`

In [ ]:
x = np.array([0.5, -1.2, 3.0])   # 3 inputs
w = np.array([0.4, 0.7, -0.2])   # 3 weights
b = 0.1                          # bias

z = np.dot(w, x) + b
print("z (weighted sum + bias):", z)
print("sigmoid(z):", sigmoid(z))
print("tanh(z)   :", tanh(z))
print("relu(z)   :", relu(z))

# Haath se check: 0.4*0.5 + 0.7*(-1.2) + (-0.2)*3.0 + 0.1
print("manual z  :", 0.4*0.5 + 0.7*(-1.2) + (-0.2)*3.0 + 0.1)

## 3. Layer = bahut se neurons ek saath

Matrix form me: `Z = X @ W + b`

Shapes (batch of `m` samples):
- `X`: (m, n_in)
- `W`: (n_in, n_out)
- `b`: (1, n_out)
- `Z`: (m, n_out)

In [ ]:
X = np.random.randn(5, 3)        # 5 samples, 3 features
W = np.random.randn(3, 4)        # 3 inputs -> 4 neurons
b = np.zeros((1, 4))

Z = X @ W + b
A = relu(Z)
print("X:", X.shape, " W:", W.shape, " Z:", Z.shape)
print(A)

## 4. 2-layer network ka forward pass

`input -> Linear -> ReLU -> Linear -> Sigmoid -> output`

In [ ]:
def init_params(n_in, n_hidden, n_out, scale=0.5):
    return {
        "W1": np.random.randn(n_in, n_hidden) * scale,
        "b1": np.zeros((1, n_hidden)),
        "W2": np.random.randn(n_hidden, n_out) * scale,
        "b2": np.zeros((1, n_out)),
    }

def forward(X, p):
    Z1 = X @ p["W1"] + p["b1"]
    A1 = relu(Z1)
    Z2 = A1 @ p["W2"] + p["b2"]
    A2 = sigmoid(Z2)
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}   # backprop me kaam aayega (Day 2)
    return A2, cache

params = init_params(n_in=2, n_hidden=8, n_out=1)
for k, v in params.items():
    print(k, v.shape)

## 5. Chhote dataset par forward pass

Abhi weights random hain, isliye network ne kuch seekha nahi. Output bas random hoga. Training Day 2 me.

In [ ]:
X, y = make_moons(n_samples=200, noise=0.2, random_state=42)
print("X:", X.shape, " y:", y.shape)

probs, cache = forward(X, params)
print("output shape:", probs.shape)
print("pehle 5 probabilities:", probs[:5].ravel().round(3))
print("min/max:", probs.min().round(3), probs.max().round(3))

preds = (probs.ravel() > 0.5).astype(int)
print("accuracy (random weights):", (preds == y).mean())

In [ ]:
# Decision regions with random weights
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
zz, _ = forward(grid, params)
zz = zz.reshape(xx.shape)

plt.figure(figsize=(6, 4))
plt.contourf(xx, yy, zz, levels=20, cmap="RdBu", alpha=0.6)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="RdBu", edgecolor="k", s=20)
plt.title("Untrained network (random weights)")
plt.colorbar(label="P(class=1)")
plt.show()

## 6. Experiments (khud karo)

1. `n_hidden` ko 2, 8, 64 karke dekho, output kaise badalta hai?
2. `relu` ki jagah `tanh` hidden layer me lagao.
3. Weights ka `scale` 0.01 aur 5 karke dekho (saturation: sigmoid 0 ya 1 par chipak jata hai).
4. Sab weights 0 kar do. Kya hota hai? (Hint: har neuron same output deta hai.)

## 7. Notes (apne words me likho)

- Neuron kya hai:
- Non-linear activation kyun zaroori hai:
- Forward pass kya karta hai: